<div style="background-color: black; color: white; padding: 10px;text-align: center;">
  <strong>Date Published:</strong> Sep 14, 2026 <strong>Author:</strong> Adnan Alaref
</div>

# Build a Transformer from Scratch in PyTorch

A step-by-step implementation of the **Transformer architecture** from the original paper, built entirely with **PyTorch** to understand how each component works internally.

### What We Build

- Multi-Head Self-Attention
- Sinusoidal Positional Encoding
- Position-Wise Feed-Forward Network
- Transformer Encoder
- Transformer Decoder
- Padding & Causal Masks
- Complete Encoder–Decoder Transformer

📖 **Original Paper:**  
[**Attention Is All You Need - Vaswani et al(2017).**](https://arxiv.org/pdf/1706.03762)

> 🎯 **Goal:** Learn the Transformer by implementing its core components from scratch rather than treating it as a black box.

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 1: Import Library.</div>

In [ ]:
import math
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

import warnings
warnings.simplefilter(action= "ignore")
warnings.filterwarnings(action= "ignore", category=FutureWarning)

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 2: Define MultiHeadAttention.</div>

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model: int, num_heads: int) -> None:
    super(MultiHeadAttention, self).__init__()

    # ── Initialize dimensions ─────────────────────────────────────────────────
    self.d_model = d_model # Model's dimension
    self.num_heads = num_heads # Number of attention heads
    self.head_dim = d_model // num_heads # Dimension of each head's key, query, and value

    # ── Scaling factor ────────────────────────────────────────────────────────
    self.scale = self.head_dim ** 0.5

    # Ensure that the model dimension (d_model) is divisible by the number of heads
    if self.head_dim * num_heads != d_model:
      raise ValueError(
        f"Embedding_dim ({d_model}) must be divisible by num heads ({num_heads})."
      )

    # ── Linear projections ────────────────────────────────────────────────────
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)

    # ── Output projection ─────────────────────────────────────────────────────
    self.fc_proj = nn.Linear(d_model, d_model)


  def split_heads(self, M: torch.Tensor)-> torch.Tensor:
    """
      Split the embedding dimension into multiple attention heads.
      Input:
          M: (B, seq_len, d_model)
      Output:
          M: (B, num_heads, seq_len, head_dim)
    """

    B, seq_len, _ = M.shape
    return M.view(
      B,
      seq_len,
      self.num_heads,
      self.head_dim).transpose(1,2)


  def combine_heads(self, M: torch.Tensor)-> torch.Tensor:
    """
      Combine multiple attention heads.
      Input:
          M: (B, num_heads, seq_len, head_dim)
      Output:
        M: (B, seq_len, d_model)
    """

    B, _, seq_len, _ = M.shape
    return M.transpose(1,2).reshape(
      B,
      seq_len,
      self.d_model
    )

  def scaled_dot_product_attention(self,
                                   query: torch.Tensor,
                                   key: torch.Tensor,
                                   value: torch.Tensor,
                                   mask: torch.Tensor | None = None,
                                   )-> torch.Tensor:

    """Compute scaled dot-product attention."""

    # ── QKᵀ / √dₖ ─────────────────────────────────────────────────────────────
    attn_scores = torch.matmul(
      query,
      key.transpose(-2,-1)
    ) / self.scale

    # ── Apply mask ────────────────────────────────────────────────────────────
    if mask is not None:
      attn_scores = attn_scores.masked_fill(
        ~mask,
        float("-inf")
      )

    # ── Convert scores → probabilities ────────────────────────────────────────
    attn_weights = F.softmax(
      attn_scores,
      dim=-1
    )

    # ── Weighted sum of values ────────────────────────────────────────────────
    output = torch.matmul(
      attn_weights,
      value
    )

    return output


  def forward(self,
              Q: torch.Tensor,
              K: torch.Tensor,
              V: torch.Tensor,
              mask: torch.Tensor | None = None
              )->torch.Tensor:
    """
      Multi-head attention.
      Architecture:
            Projection → Split → Attention → Combine → Projection
    """

    # ── Linear projections ────────────────────────────────────────────────────
    query = self.W_q(Q)
    key = self.W_k(K)
    value = self.W_v(V)

    # ── Split into attention heads ────────────────────────────────────────────
    query = self.split_heads(query)
    key = self.split_heads(key)
    value = self.split_heads(value)

    # ── Scaled dot-product attention ──────────────────────────────────────────
    attn_output = self.scaled_dot_product_attention(
      query,
      key,
      value,
      mask
    )

    # ── Combine heads ─────────────────────────────────────────────────────────
    output = self.combine_heads(
      attn_output
    )

    # ── Output projection ─────────────────────────────────────────────────────
    return self.fc_proj(
      output
    )

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 3: Define PositionalEncoding.</div>

In [ ]:
class PositionalEncoding(nn.Module):
  """
    Generate sinusoidal positional encodings for Transformer inputs.

    PE(pos, 2i)   = sin(pos / 10000^(2i / d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))
  """

  def __init__(self, max_seq_len: int, d_model: int) -> None:
    super(PositionalEncoding, self).__init__()

    if d_model % 2 != 0:
      raise ValueError(
        "d_model must be even to pair sine and cosine dimensions "
        "as in the original Transformer paper."
      )

    PE = torch.zeros(max_seq_len, d_model)

    pos = torch.arange(max_seq_len)[:, None]

    i = torch.arange(0, d_model, 2)

    div_term = torch.exp(-( i / d_model) * math.log(10_000))

    angle = pos * div_term

    PE[:, 0::2] = torch.sin(angle)
    PE[:, 1::2] = torch.cos(angle)

    self.register_buffer("PE", PE)

  def forward(self, x: torch.Tensor)-> torch.Tensor:
    seq_len = x.shape[1]
    return x + self.PE[:seq_len]

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 4: Define PositionWiseFeedForward.</div>

In [ ]:
class PositionWiseFeedForward(nn.Module):
  """
    Position-wise feed-forward network used in the Transformer.

    Applies two linear projections with a non-linear activation
    independently to each token position.
  """

  def __init__(self, d_model: int, expansion_dim: int) -> None:
    super().__init__()

    self.fc1_proj = nn.Linear(d_model, expansion_dim)
    self.fc2_proj = nn.Linear(expansion_dim, d_model)
    self.relu = nn.ReLU()

  def forward(self, x:torch.Tensor)-> torch.Tensor:
    return self.fc2_proj(
      self.relu(
        self.fc1_proj(x)
      )
    )

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 5: Build EncoderLayer.</div>

<p align="center">
  <img src="https://media.datacamp.com/legacy/v1691083306/Figure_2_The_Encoder_part_of_the_transformer_network_Source_image_from_the_original_paper_b0e3ac40fa.png"
       alt="Transformer Architecture"
       width="200"
       height="150">
</p>

In [ ]:
class EncoderLayer(nn.Module):
  """
    Transformer encoder layer using Post-LayerNorm.

    Steps:
       1. Self-attention:
           x = LayerNorm(x + Dropout(Self-Attention(x)))

       2. Position-wise feed-forward network:
           x = LayerNorm(x + Dropout(FFN(x)))
  """

  def __init__(self,
               d_model: int,
               num_heads: int,
               expansion_dim: int,
               dropout_prop: float) -> None:

    super().__init__()

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)

    self.dropout = nn.Dropout(dropout_prop)

    self.self_attn = MultiHeadAttention(d_model, num_heads)
    self.feed_forward = PositionWiseFeedForward(d_model, expansion_dim)

  def forward(
      self,
      x: torch.Tensor,
      mask: torch.Tensor | None = None
    )-> torch.Tensor:

    # ── Self-Attention ──
    attn_output = self.self_attn(x, x, x, mask)
    x = self.norm1(x + self.dropout(attn_output))

    # ── Feed-Forward Network ──
    ffn_output = self.feed_forward(x)
    x = self.norm2(x + self.dropout(ffn_output))

    return x

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 6: Build DecoderLayer.</div>

<p align="center">
  <img src="https://media.datacamp.com/legacy/v1691083444/Figure_3_The_Decoder_part_of_the_Transformer_network_Souce_Image_from_the_original_paper_b90d9e7f66.png"
       alt="Transformer Architecture"
       width="150"
       height="150">
</p>

In [ ]:
class DecoderLayer(nn.Module):
  """
    Transformer decoder layer using Post-LayerNorm.

    Steps:
        1. Masked self-attention:
           x = LayerNorm(x + Dropout(Self-Attention(x, x, x, tgt_mask)))

        2. Cross-attention:
           x = LayerNorm(x + Dropout(Cross-Attention(x, enc_output, enc_output, src_mask)))

        3. Position-wise feed-forward network:
           x = LayerNorm(x + Dropout(FFN(x)))
  """
  def __init__(self,
               d_model: int,
               num_heads: int,
               expansion_dim: int,
               dropout_prop: float) -> None:
    super().__init__()

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)

    self.dropout = nn.Dropout(dropout_prop)

    self.self_attn = MultiHeadAttention(d_model, num_heads)
    self.cross_attn = MultiHeadAttention(d_model, num_heads)
    self.feed_forward = PositionWiseFeedForward(d_model, expansion_dim)

  def forward(
      self,
      x: torch.Tensor,
      enc_output: torch.Tensor,
      src_mask: torch.Tensor,
      tgt_mask: torch.Tensor
    )-> torch.Tensor:

    # ── Masked Self-Attention ──
    attn_output = self.self_attn(x, x, x, tgt_mask)
    x = self.norm1(x + self.dropout(attn_output))

    # ── Cross-Attention ──
    attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
    x = self.norm2(x + self.dropout(attn_output))

    # ── Feed-Forward Network ──
    ffn_output = self.feed_forward(x)
    x = self.norm3(x + self.dropout(ffn_output))

    return x

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 7: Create The Complete Transformer Network.</div>

<center
![](https://media.datacamp.com/legacy/v1691083566/Figure_4_The_Transformer_Network_Source_Image_from_the_original_paper_120e177956.png)
\>

<p align="center">
  <img 
 src="https://media.datacamp.com/legacy/v1691083566/Figure_4_The_Transformer_Network_Source_Image_from_the_original_paper_120e177956.png"
       alt="Transformer Architecture"
       width="200"
       height="150">
</p>

In [ ]:
class Transformer(nn.Module):
  def __init__(
      self,
      src_vocab_size: int,
      tgt_vocab_size: int,
      max_seq_len: int,
      d_model: int,
      expansion_dim: int,
      num_heads: int,
      num_layers: int,
      dropout: float,
      pad_idx: int,
    ) -> None:

    super().__init__()

    self.pad_idx = pad_idx
    self.encoder_embedding = nn.Embedding(
      src_vocab_size,
      d_model,
      padding_idx=pad_idx
    )

    self.decoder_embedding = nn.Embedding(
      tgt_vocab_size,
      d_model,
      padding_idx=pad_idx
    )

    # ── Positional Encoding ───────────────────────────────────────────
    # The same max_seq_len is used for both source and target sequences,
    # so a single positional encoding module can be shared by both.
    self.positional_encoding = PositionalEncoding(
      max_seq_len,
      d_model
    )

    self.encoder_layers = nn.ModuleList([
      EncoderLayer(
        d_model,
        num_heads,
        expansion_dim,
        dropout
      )
      for _ in range(num_layers)
    ])

    self.decoder_layers = nn.ModuleList([
      DecoderLayer(
        d_model,
        num_heads,
        expansion_dim,
        dropout
      )
      for _ in range(num_layers)
    ])

    self.fc_proj = nn.Linear(
      d_model,
      tgt_vocab_size
    )

    self.dropout = nn.Dropout(
      dropout
    )

  def create_masks(self,
                   src: torch.Tensor,
                   tgt: torch.Tensor,
                  )->tuple[torch.Tensor, torch.Tensor]:
    """
      Create attention masks for an encoder-decoder Transformer.

      The source padding mask prevents attention to <PAD> tokens in:
          1. Encoder self-attention.
          2. Decoder cross-attention.

      The target mask combines:
          1. Target padding mask: prevents attention to <PAD> tokens.
          2. Target causal mask: prevents attention to future tokens.

      Args:
          src: Source token IDs of shape (batch_size, src_len).
          tgt: Target token IDs of shape (batch_size, tgt_len).
          pad_idx: Token ID used to represent <PAD> tokens.

      Returns:
          A tuple containing:
              src_padding_mask:
                  Boolean mask of shape (batch_size, 1, src_len).
                    1- True  → valid token
                    2- False → <PAD>

              tgt_mask:
                  Boolean mask of shape (batch_size, tgt_len, tgt_len).
                    1- True  → current/past token is visible
                    2- False → future token is hidden or a <PAD> token.
    """

    # ── Padding masks ───────────────────────────────────────
    src_padding_mask = (src != self.pad_idx)[:,None,:]
    tgt_padding_mask = (tgt != self.pad_idx)[:,None,:]

    # ── Causal mask ─────────────────────────────────────────
    tgt_len = tgt.shape[1]

    tgt_causal_mask = torch.tril(
      torch.ones(
        1,
        tgt_len,
        tgt_len,
        dtype=torch.bool,
        device= tgt.device
      )
    )

    # ── Combined target mask ────────────────────────────────
    tgt_mask = tgt_causal_mask & tgt_padding_mask

    return src_padding_mask, tgt_mask

  def forward(self,
              src:torch.Tensor,
              tgt: torch.Tensor
             ) -> torch.Tensor:
    """
      Args:
          src: Source token IDs, shape (B, S).
          tgt: Target token IDs, shape (B, T).

      Returns:
        Output logits of shape (B, T, tgt_vocab_size).
    """

    src_mask, tgt_mask = self.create_masks(src, tgt, self.pad_idx)

    # ── Embedding + Positional Encoding ──────────────────
    src_embedded = self.dropout(
      self.positional_encoding(
        self.encoder_embedding(src)
      )
    )

    tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

    # ── Encoder ──────────────────────────────────────────
    encoder_output = src_embedded

    for enc_layer in self.encoder_layers:
      encoder_output = enc_layer(
        encoder_output,

        # Source padding mask.
        # Used by self-attention to prevent the encoder
        # from attending to <PAD> tokens in the source.
        src_mask
      )

    # ── Decoder ──────────────────────────────────────────
    decoder_output = tgt_embedded

    for dec_layer in self.decoder_layers:
      decoder_output = dec_layer(
        decoder_output,
        # same final encoder representation
        encoder_output,

        # Source padding mask.
        # Used by cross-attention to prevent the decoder
        # from attending to <PAD> tokens in the source.
        src_mask,

        # Target mask = padding mask + causal mask.
        # Used by decoder self-attention to prevent:
        # 1. attending to <PAD> tokens
        # 2. attending to future target tokens
        tgt_mask
      )

    # ── Output Projection ────────────────────────────────
    output = self.fc_proj(decoder_output)

    return output

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Thanks & Upvote ❤️</div>